# S32_05 — SHAP: Model Explainability for Trees and Pipelines

## Why SHAP?

Tree-based `.feature_importances_` tells you which features were used most across splits — but it doesn't tell you *how* each feature affected any individual prediction, and it can be biased toward high-cardinality features.

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution value for each prediction, grounded in cooperative game theory. Key properties:
- **Consistent:** if a feature has more impact in model B than A, its SHAP value will always be higher in B
- **Additive:** SHAP values sum to the prediction (relative to the base rate)
- **Local + global:** explain a single prediction *or* aggregate across the dataset

## Installation

```bash
pip install shap
```

## TreeExplainer — fast path for tree models

`shap.TreeExplainer` works natively with sklearn trees, Random Forests, GradientBoostingClassifier, XGBoost, LightGBM, and CatBoost.

In [ ]:
import shap
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

## Global feature importance — summary plot

In [ ]:
shap.summary_plot(shap_values, X_test)

Each dot is one sample. Colour = feature value (red = high, blue = low). X-axis = SHAP value (positive = pushes toward class 1).

## Local explanation — single prediction

In [ ]:
# Explain the first test sample
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0],
    matplotlib=True
)

## Bar plot — mean absolute SHAP (global ranking)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type='bar')

## Using SHAP with a sklearn Pipeline

When a pipeline includes preprocessing steps, pass the raw (pre-transformed) data to a `KernelExplainer`, or extract the fitted estimator and pass transformed data to `TreeExplainer`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
pipe.fit(X_train, y_train)

# Transform data then explain the fitted estimator directly
X_test_transformed = pipe[:-1].transform(X_test)
explainer_pipe = shap.TreeExplainer(pipe['clf'])
shap_values_pipe = explainer_pipe.shap_values(X_test_transformed)

shap.summary_plot(shap_values_pipe, X_test_transformed, feature_names=X_test.columns.tolist())

## KernelExplainer — model-agnostic (slower)

Use `KernelExplainer` for models that `TreeExplainer` doesn't support (SVMs, logistic regression, etc.). It's slower — use a background sample.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000).fit(X_train, y_train)

background = shap.sample(X_train, 100)  # summarise background with 100 samples
kernel_explainer = shap.KernelExplainer(lr.predict_proba, background)
shap_values_lr = kernel_explainer.shap_values(X_test.iloc[:50])  # subset for speed

shap.summary_plot(shap_values_lr[1], X_test.iloc[:50])  # class 1 SHAP values

## Quick reference

| Model type | Recommended explainer | Speed |
|---|---|---|
| sklearn trees, RF, GBM | `TreeExplainer` | Fast |
| XGBoost, LightGBM, CatBoost | `TreeExplainer` | Fast |
| Linear models | `LinearExplainer` | Fast |
| SVM, any black-box | `KernelExplainer` | Slow |

**Further reading:** [shap.readthedocs.io](https://shap.readthedocs.io)
